In [1]:
import os
import pandas as pd
import torch
from PIL import Image
from torch.backends import cudnn
from torch.utils.data import DataLoader
from torchvision.datasets import VisionDataset
from torchvision.transforms import InterpolationMode, v2
from torchvision.models import vit_b_16, ViT_B_16_Weights

torch.manual_seed(3407)
torch.cuda.manual_seed(3407)
cudnn.deterministic = True
cudnn.benchmark = False
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)



Device: cuda


In [2]:
# Paths
train_csv_path = "/kaggle/input/cassava-leaf-disease-classification/train.csv"
train_img_dir = "/kaggle/input/cassava-leaf-disease-classification/train_images/"
test_dir = "/kaggle/input/cassava-leaf-disease-classification/test_images/"

# Hyper‑parameters
img_size = 224
batch_size = 16
num_workers = 4
num_classes = 5
tta = False
epochs = 3
lr = 1e-4

# Model
vit_model = vit_b_16(weights=ViT_B_16_Weights.IMAGENET1K_V1)
if isinstance(vit_model.heads, torch.nn.Sequential):
    last_linear = vit_model.heads[-1]
    vit_model.heads[-1] = torch.nn.Linear(last_linear.in_features, num_classes)
else:
    vit_model.heads = torch.nn.Linear(vit_model.heads.in_features, num_classes)
vit_model = vit_model.to(device)




Downloading: "https://download.pytorch.org/models/vit_b_16-c867db91.pth" to /root/.cache/torch/hub/checkpoints/vit_b_16-c867db91.pth
100%|██████████| 330M/330M [00:03<00:00, 115MB/s]


In [3]:
class CassavaTrainDataset(VisionDataset):
    """Dataset for training images with labels."""

    def __init__(self, csv_path, img_dir, transform=None):
        super().__init__(root=img_dir)
        self.df = pd.read_csv(csv_path)
        self.transform = transform
        self.img_dir = img_dir

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img_path = os.path.join(self.img_dir, row["image_id"])
        img = Image.open(img_path).convert("RGB")
        if self.transform:
            img = self.transform(img)
        label = int(row["label"])
        return img, label

    def __len__(self):
        return len(self.df)


# Training augmentations
train_transforms = v2.Compose(
    [
        v2.RandomResizedCrop((img_size, img_size), scale=(0.8, 1.0)),
        v2.RandomHorizontalFlip(),
        v2.ToImage(),
        v2.ToDtype(torch.float32, scale=True),
        v2.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ]
)

train_dataset = CassavaTrainDataset(
    train_csv_path, train_img_dir, transform=train_transforms
)
train_loader = DataLoader(
    train_dataset,
    batch_size=batch_size,
    shuffle=True,
    num_workers=num_workers,
    pin_memory=True,
)

criterion = torch.nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(vit_model.parameters(), lr=lr)



In [4]:
# Simple training loop
vit_model.train()
for epoch in range(epochs):
    running_loss = 0.0
    for imgs, labels in train_loader:
        imgs = imgs.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()
        outputs = vit_model(imgs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * imgs.size(0)

    epoch_loss = running_loss / len(train_loader.dataset)
    print(f"Epoch [{epoch+1}/{epochs}] - Loss: {epoch_loss:.4f}")



Epoch [1/3] - Loss: 0.6609
Epoch [2/3] - Loss: 0.5540
Epoch [3/3] - Loss: 0.5141


In [5]:
# Test‑time transforms (same as original)
test_transforms = v2.Compose(
    [
        v2.Resize((img_size, img_size), interpolation=InterpolationMode.BICUBIC),
        v2.CenterCrop((img_size, img_size)),
        v2.ToImage(),
        v2.ToDtype(torch.float32, scale=True),
        v2.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ]
)

if tta:
    ttas = [
        v2.RandomResizedCrop((img_size, img_size), (0.5, 1)),
        v2.RandomRotation(180),
        v2.RandomVerticalFlip(1),
        v2.RandomAffine(180),
        v2.RandomPerspective(p=1),
    ]
else:
    ttas = None


class CassavaTestDataset(VisionDataset):
    """Dataset for the Cassava test images (no labels)."""

    def __init__(self, data_dir, transform=None, ttas=None):
        super().__init__(root=data_dir)
        self.transform = transform
        self.images = sorted(
            [
                f
                for f in os.listdir(data_dir)
                if os.path.isfile(os.path.join(data_dir, f))
                and f.lower().endswith((".png", ".jpg", ".jpeg"))
            ]
        )
        self.ttas = ttas

    def __getitem__(self, idx):
        filename = self.images[idx]
        img_path = os.path.join(self.root, filename)
        img = Image.open(img_path).convert("RGB")
        if self.ttas is not None and self.transform is not None:
            img = [self.transform(t(img)) for t in self.ttas]
        elif self.transform:
            img = self.transform(img)
        return img, filename

    def __len__(self):
        return len(self.images)


test_dataset = CassavaTestDataset(test_dir, transform=test_transforms, ttas=ttas)
test_loader = DataLoader(
    test_dataset,
    batch_size=batch_size,
    shuffle=False,
    num_workers=num_workers,
    pin_memory=True,
)

normalizer = torch.nn.Softmax(dim=1)



In [6]:
# Inference
vit_model.eval()
all_names = []
all_preds = []
with torch.no_grad():
    for inputs, filenames in test_loader:
        if tta:
            # inputs is a list of augmented tensors
            inputs = torch.cat(inputs, dim=0).to(device)
            preds = normalizer(vit_model(inputs))
            batch_preds = torch.stack(torch.split(preds, len(filenames)), dim=0)
            mean_preds = torch.mean(batch_preds, dim=0)
            pred_labels = torch.argmax(mean_preds, dim=1).tolist()
        else:
            inputs = inputs.to(device)
            preds = normalizer(vit_model(inputs))
            pred_labels = torch.argmax(preds, dim=1).tolist()
        all_names.extend(filenames)
        all_preds.extend(pred_labels)



In [7]:
# Create submission file
submission = pd.DataFrame({"image_id": all_names, "label": all_preds})
submission.to_csv("submission.csv", index=False)
print("Submission saved to submission.csv")

Submission saved to submission.csv
